# Лабораторная работа №1

Юдин Артём М4221

Необходимо обучить классификатор поверх текстовых эмбедингов. Я взял датасет с джейлбрекй промптами, будет подобие safe guard модели.

In [37]:
import torch
import copy
from torch import nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer, set_seed
from sklearn.metrics import f1_score, precision_score, recall_score
from tqdm.auto import tqdm

In [38]:
DATASET_NAME = "jackhhao/jailbreak-classification"
MODEL_NAME = "distilbert/distilbert-base-uncased"
TEXT_COLUMN = "prompt"
LABEL_COLUMN = "type"
MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCHS = 5
LR = 2e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_LABELS = 2
PRINT_WIDTH = 62

SEED = 42
set_seed(SEED, deterministic=True)

Загружаем датасет, делаем 2 сплита

In [39]:
def map_label(exp):
    mapping = {'benign': 0, "jailbreak": 1}
    exp[LABEL_COLUMN] = mapping[exp[LABEL_COLUMN]]
    return exp

dataset = load_dataset(DATASET_NAME)
dataset = dataset.map(map_label)
train_val_split = dataset["train"].train_test_split(test_size=0.3)
train_split = train_val_split["train"]
val_split = train_val_split["test"]
test_split = dataset["test"]

Токенизируем текст

In [40]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def collate(batch):
    encoded = tokenizer(
        [row[TEXT_COLUMN] for row in batch],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    encoded["labels"] = torch.tensor([row[LABEL_COLUMN] for row in batch])
    return encoded


train_loader = DataLoader(train_split, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_split, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_split, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

Пишем пуллинги и класс

In [41]:
def mean_pooling(hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
    return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)


def class_pooling(hidden, attention_mask):
    return hidden[:, 0]


class TextClassifier(nn.Module):
    def __init__(self, encoder, num_labels, pooling):
        super().__init__()
        self.encoder = encoder
        self.pooling = pooling
        self.head = nn.Linear(encoder.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        return self.head(self.pooling(hidden, attention_mask))

In [42]:
def train(model, train_loader, val_loader, epochs, lr, device, save_path="best.pt"):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    model.train()

    best_recall = 0.0
    best_model_weights = None

    for epoch in range(epochs):
        total = 0.0
        for batch in tqdm(train_loader, desc="Training", leave=False):
            labels = batch.pop("labels").to(device)
            inputs = {key: value.to(device) for key, value in batch.items()}
            loss = loss_fn(model(inputs["input_ids"], inputs["attention_mask"]), labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item()
        val_metrics = test(model, val_loader, device)
        val_recall = val_metrics["recall"]

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Loss: {total / len(train_loader):.4f} | "
            f"Val F1: {val_metrics["f1"]:.4f} | "
            f"Val Precision: {val_metrics['precision']:.4f} | "
            f"Val Recall: {val_metrics['recall']:.4f}"
        )
        if val_recall > best_recall:
            best_recall = val_recall
            best_model_weights = copy.deepcopy(model.state_dict())
            torch.save(best_model_weights, save_path)
            print(f" -> Saved new best model with Recall: {best_recall:.4f}")

    if best_model_weights is not None:
        model.load_state_dict(best_model_weights)
        print(f"\nLoaded best model parameters (Val Recall: {best_recall:.4f}).")
    return model


def train_mean_pooling(train_loader, val_loader, num_labels, epochs=EPOCHS, lr=LR, device=DEVICE):
    encoder = AutoModel.from_pretrained(MODEL_NAME)
    return train(TextClassifier(encoder, num_labels, mean_pooling), train_loader, val_loader, epochs, lr, device)


def train_class_pooling(train_loader, val_loader, num_labels, epochs=EPOCHS, lr=LR, device=DEVICE):
    encoder = AutoModel.from_pretrained(MODEL_NAME)
    return train(TextClassifier(encoder, num_labels, class_pooling), train_loader, val_loader, epochs, lr, device)

In [43]:
@torch.no_grad()
def test(model, loader, device=DEVICE):
    model.eval()
    model.to(device)
    predictions, targets = [], []
    for batch in tqdm(loader, desc="Testing", leave=False):
        labels = batch.pop("labels")
        inputs = {key: value.to(device) for key, value in batch.items()}
        predictions.extend(model(inputs["input_ids"], inputs["attention_mask"]).argmax(dim=-1).cpu().tolist())
        targets.extend(labels.tolist())
    average = "binary" if NUM_LABELS == 2 else "macro"
    return {
        "precision": precision_score(targets, predictions, average=average, zero_division=0),
        "recall": recall_score(targets, predictions, average=average, zero_division=0),
        "f1": f1_score(targets, predictions, average=average, zero_division=0),
    }

In [44]:
mean_model = train_mean_pooling(train_loader, val_loader, NUM_LABELS)
test_metrics = test(mean_model, test_loader)
print("="*PRINT_WIDTH)
print(
    f"Test F1: {test_metrics["f1"]:.4f} | "
    f"Test Precision: {test_metrics['precision']:.4f} | "
    f"Test Recall: {test_metrics['recall']:.4f}"
)
print("="*PRINT_WIDTH)

class_model = train_class_pooling(train_loader, val_loader, NUM_LABELS)
test_metrics = test(class_model, test_loader)
print("="*PRINT_WIDTH)
print(
    f"Test F1: {test_metrics["f1"]:.4f} | "
    f"Test Precision: {test_metrics['precision']:.4f} | "
    f"Test Recall: {test_metrics['recall']:.4f}"
)
print("="*PRINT_WIDTH)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Training:   0%|          | 0/23 [00:00<?, ?it/s]

Testing:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1/5 | Train Loss: 0.3951 | Val F1: 0.9550 | Val Precision: 0.9787 | Val Recall: 0.9324
 -> Saved new best model with Recall: 0.9324


...

Training:   0%|          | 0/23 [00:00<?, ?it/s]

Testing:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 5/5 | Train Loss: 0.0027 | Val F1: 0.9793 | Val Precision: 1.0000 | Val Recall: 0.9595

Loaded best model parameters (Val Recall: 0.9662).


Testing:   0%|          | 0/9 [00:00<?, ?it/s]

Test F1: 0.9714 | Test Precision: 0.9645 | Test Recall: 0.9784


# Выводы

По итогам работы мы обучили 2 классификатора поверх distilbert модели. На отложенной тестовой выборке лучше всего себя показал вариат с пуллингом класса, однако разница в метриках там незначительная, так что можно сказать, что оба варианта пулинга на нашем наборе данных отработали одинаково